# v1.All_tracks_v3 — spatial-bin analysis of the 1993-2013 production run

Re-does the *bin analysis* of `Analysis/All_tracks_MERCATOR.ipynb` for the new
1993-2013 run (`run_2/tracks_v3_*`), **all four seeds combined into one
21-year climatology**.

Method is the same as the template (xhistogram-style lon/lat counts of particle
positions, grouped by release month / release season, obs aggregated into 5-day
bins, plus a mean-particle-age map), but the computation is **streamed store by
store** instead of concatenating all 1534 zarr stores: each store is read once in
a worker process, binned with `xhistogram` into a small `(obs_bin, lon, lat)`
count array, and the result is accumulated into one master array

    H[release_year, release_month, obs_bin, lon, lat]

from which the monthly and seasonal histograms are just sums (histogram counts
are additive, so this is identical to binning the concatenated dataset). Memory
stays flat (~1 GB for the master array + ~1 GB per worker).

## Spatial grid — shared by every product in `run_2/analysis`

The bin **edges** are the template's, built from the data extremes of the run:

```python
lat_bins = np.linspace(lat_min, lat_max, 30)
lon_bins = np.linspace(lon_min, lon_max, 80)
```

i.e. 79 x 29 cells of about 1.04 deg lon x 1.26 deg lat over
97.0W-14.7W, 4.4S-32.0N. Those edges were computed once by the `v2.*`
notebooks and saved to **`v2.bins_v3.npz`**, which is loaded here rather than
recomputed, so the histograms, the monthly mean fields (`mean_*_month_v3.zarr`)
and every figure in this folder sit on exactly **one common grid**.
(The earlier 0.5 deg `arange` grid is obsolete and its files have been
replaced.)

In [1]:
# ---- papermill parameters -------------------------------------------------
tracks_dirs = [
    "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_11098723",
    "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_45678",
    "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_67891",
    "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_78876",
]
out_dir = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis"

# spatial bins: the shared grid of run_2/analysis, edges as in the template
#   lat_bins = np.linspace(lat_min, lat_max, 30)
#   lon_bins = np.linspace(lon_min, lon_max, 80)
# computed from the data extremes by the v2 notebooks and stored here:
bins_file = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/v2.bins_v3.npz"

# obs handling: skip the release position (obs 0), then group obs into 5-day
# bins (6-hourly output -> 4*5 = 20 obs per bin), trimming the tail.
obs_start    = 1
# 6-hourly output, 4*5 obs per 5-day bin
obs_coarsen  = 20
# 36 bins x 20 obs = 720 obs = 180 days after release
n_obs_bins   = 36

# processes reading/binning stores in parallel
n_workers  = 48
# 0 means all stores, a positive number processes only the first N (for testing)
max_stores = 0

out_month      = "_hist_month_v3.nc"
out_season     = "_hist_season_v3.nc"
out_year_month = "_hist_year_month_v3.nc"
out_age        = "_hist_age_month_v3.nc"

In [2]:
# Parameters
n_workers = 48


In [3]:
import os, sys, time, json, glob
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import xarray as xr
from xhistogram.xarray import histogram as xhist

t_start = time.time()
print("python :", sys.executable)
import xhistogram
print("xarray :", xr.__version__, "| numpy :", np.__version__,
      "| xhistogram :", xhistogram.__version__)
os.makedirs(out_dir, exist_ok=True)

python : /work/bk1450/b383184/conda/envs/parcels_3.1.2/bin/python
xarray : 2025.6.1 | numpy : 2.2.6 | xhistogram : 0.3.2


## Store list

In [4]:
stores = []
for d in tracks_dirs:
    s = sorted(Path(d).glob("Parcels_run_*.zarr"))
    print(f"{d:<80s} {len(s):5d} stores")
    stores += s
if max_stores:
    stores = stores[:max_stores]
print("total stores to process:", len(stores))
print(stores[0].name, "...", stores[-1].name)

/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_11098723            438 stores
/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_45678               439 stores
/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_67891               292 stores
/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tracks_v3_78876               365 stores
total stores to process: 1534
Parcels_run_11098723_1993-01-06 00:00:00.zarr ... Parcels_run_78876_2013-12-28 00:00:00.zarr


## Bins

`lon_bins` / `lat_bins` are the **edges**, loaded from `v2.bins_v3.npz` so that
this notebook, the `v2.*` monthly-mean notebooks and all the figures use the
identical grid (80 lon edges, 30 lat edges from `np.linspace` over the data
extremes, exactly as in `Analysis/All_tracks_MERCATOR.ipynb`). They are passed
to `xhist(..., bins=[lon_bins, lat_bins])`; the saved `lon`/`lat` coordinates
are the bin centres, again as xhistogram does it.

In [5]:
_b = np.load(bins_file)
lon_bins = _b["lon_bins"].astype("float64")
lat_bins = _b["lat_bins"].astype("float64")
lon_c = 0.5*(lon_bins[:-1] + lon_bins[1:])
lat_c = 0.5*(lat_bins[:-1] + lat_bins[1:])
n_lon, n_lat = lon_c.size, lat_c.size
dlon = float(np.diff(lon_bins).mean())
dlat = float(np.diff(lat_bins).mean())
print("bin edges from:", bins_file)
print(f"lon: {lon_bins.size} edges -> {n_lon} bins of {dlon:.4f} deg  "
      f"[{lon_bins[0]:.4f}, {lon_bins[-1]:.4f}]")
print(f"lat: {lat_bins.size} edges -> {n_lat} bins of {dlat:.4f} deg  "
      f"[{lat_bins[0]:.4f}, {lat_bins[-1]:.4f}]")
print("obs bins:", n_obs_bins, f"({obs_coarsen} obs = {obs_coarsen/4:.0f} days each), "
      f"obs {obs_start}..{obs_start + n_obs_bins*obs_coarsen - 1}")

years  = np.arange(1993, 2014)
months = np.arange(1, 13)
seasons = ["JFM", "AMJ", "JAS", "OND"]   # JFM transition, AMJ wet, JAS transition, OND dry
month_to_season = {1:"JFM",2:"JFM",3:"JFM", 4:"AMJ",5:"AMJ",6:"AMJ",
                   7:"JAS",8:"JAS",9:"JAS", 10:"OND",11:"OND",12:"OND"}

bin edges from: /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/v2.bins_v3.npz
lon: 80 edges -> 79 bins of 1.0414 deg  [-96.9413, -14.6725]
lat: 30 edges -> 29 bins of 1.2555 deg  [-4.3848, 32.0233]
obs bins: 36 (20 obs = 5 days each), obs 1..720


## Per-store worker

The binning itself is done by `xhistogram` exactly as in the template
(`calc_hist` / `calc_age` below are the template's functions); the only change is
that they are applied to **one store at a time** and the resulting count arrays
are summed, which is exact because histogram counts are additive.

For one zarr store the worker returns

* `hist`  — `(n_obs_bins, n_lon, n_lat)` counts of particle positions, obs 1..720
  aggregated into 5-day bins (`calc_hist` + `coarsen(obs=4*5).sum()`);
* `age_sum`, `age_cnt` — `(n_lon, n_lat)`, the template's `calc_age` summed over
  obs: age-weighted counts and counts per cell -> mean age map;
* counters used for the verification at the end.

All particles of one store share the same release time (`time[:,0]`), so the
release year/month is a property of the store.

In [6]:
def calc_hist(d):
    """template's histogram: lon/lat counts per obs step"""
    return xhist(d.lon, d.lat, bins=[lon_bins, lat_bins],
                 dim=["trajectory"], bin_dim_suffix="")


def calc_age(d):
    """template's calc_age: counts and age-weighted counts per obs step"""
    count   = xhist(d.lon, d.lat, bins=[lon_bins, lat_bins],
                    dim=["trajectory"], bin_dim_suffix="")
    age_sum = xhist(d.lon, d.lat, bins=[lon_bins, lat_bins],
                    dim=["trajectory"], weights=d.age_days, bin_dim_suffix="")
    return xr.Dataset({"count": count, "age_sum": age_sum})


def process_store(path):
    import numpy as np, pandas as pd, xarray as xr
    from xhistogram.xarray import histogram as xhist

    ds = xr.open_zarr(path)
    n_traj, n_obs = ds.lon.shape
    n_use = min(n_obs, obs_start + n_obs_bins*obs_coarsen)
    ds = ds[["lon", "lat", "time"]].isel(obs=slice(0, n_use)).load()

    t0 = ds.time.isel(obs=0)
    t0v = t0.values[~np.isnat(t0.values)]
    release = pd.Timestamp(t0v[0]) if t0v.size else pd.NaT
    n_start_times = int(np.unique(t0v).size)

    # days since each particle's release, from the actual output times
    ds = ds.assign(age_days=(ds.time - t0)/np.timedelta64(1, "D"))

    # drop the release position (obs 0), as in the template
    d = ds.isel(obs=slice(obs_start, None))
    n_valid = int(np.isfinite(d.lon.values).sum())

    # --- counts, aggregated into 5-day obs bins (template: coarsen(obs=4*5).sum())
    hist = calc_hist(d).coarsen(obs=obs_coarsen, boundary="trim").sum()
    hist = hist.isel(obs=slice(0, n_obs_bins))
    hist = hist.transpose("obs", "lon", "lat").values

    # --- mean-age map (template: calc_age(...).sum("obs"))
    age = calc_age(d).sum("obs")
    age_cnt = age["count"].transpose("lon", "lat").values
    age_sum = age["age_sum"].transpose("lon", "lat").values

    return dict(path=str(path), release=release, n_traj=int(n_traj), n_obs=int(n_obs),
                n_obs_used=int(d.sizes["obs"]), n_valid=n_valid,
                n_inside=float(age_cnt.sum()), n_binned=float(hist.sum()),
                n_start_times=n_start_times,
                hist=hist.astype(np.float64), age_sum=age_sum.astype(np.float64),
                age_cnt=age_cnt.astype(np.float64))

## Run: stream all stores, accumulate

In [7]:
H = np.zeros((years.size, months.size, n_obs_bins, n_lon, n_lat), dtype=np.float64)
A_sum = np.zeros((months.size, n_lon, n_lat), dtype=np.float64)
A_cnt = np.zeros((months.size, n_lon, n_lat), dtype=np.float64)

rows = []
tot_traj = tot_valid = tot_inside = tot_binned = 0
t0_run = time.time()

with ProcessPoolExecutor(max_workers=n_workers) as ex:
    for k, r in enumerate(ex.map(process_store, stores, chunksize=1), 1):
        rel = r["release"]
        iy = int(rel.year) - int(years[0])
        im = int(rel.month) - 1
        H[iy, im] += r["hist"]
        A_sum[im] += r["age_sum"]
        A_cnt[im] += r["age_cnt"]
        tot_traj   += r["n_traj"]
        tot_valid  += r["n_valid"]
        tot_inside += r["n_inside"]
        tot_binned += r["n_binned"]
        rows.append({k2: r[k2] for k2 in
                     ("path","release","n_traj","n_obs","n_obs_used","n_valid",
                      "n_inside","n_binned","n_start_times")})
        if k % 50 == 0 or k == len(stores):
            el = time.time() - t0_run
            print(f"{k:5d}/{len(stores)}  {el/60:6.1f} min elapsed, "
                  f"{el/k*(len(stores)-k)/60:6.1f} min left, "
                  f"{tot_traj:,} trajectories", flush=True)

log = pd.DataFrame(rows)
print("\nwall time for the streaming pass: %.1f min" % ((time.time()-t0_run)/60))

   50/1534     0.2 min elapsed,    5.9 min left, 503,050 trajectories


  100/1534     0.2 min elapsed,    2.9 min left, 1,006,100 trajectories


  150/1534     0.2 min elapsed,    2.2 min left, 1,509,150 trajectories


  200/1534     0.3 min elapsed,    2.0 min left, 2,012,200 trajectories


  250/1534     0.3 min elapsed,    1.8 min left, 2,515,250 trajectories


  300/1534     0.4 min elapsed,    1.6 min left, 3,018,300 trajectories


... (22 outputs left out)

 1450/1534     1.6 min elapsed,    0.1 min left, 14,636,683 trajectories


 1500/1534     1.7 min elapsed,    0.0 min left, 15,141,733 trajectories


 1534/1534     1.7 min elapsed,    0.0 min left, 15,485,167 trajectories



wall time for the streaming pass: 1.7 min


## Verification

In [8]:
n_stores = len(stores)
print(f"stores processed              : {n_stores}")
print(f"trajectories read             : {tot_traj:,}")
print(f"stores with >1 release time   : {int((log.n_start_times > 1).sum())}")
print(f"obs lengths on disk           : {sorted(log.n_obs.unique().tolist())}")
print(f"obs steps used per store      : {sorted(log.n_obs_used.unique().tolist())}")
print()
print(f"finite (lon,lat) positions read (obs>={obs_start})     : {tot_valid:,}")
print(f"  of these inside the lon/lat bins (counted by xhist): {tot_inside:,.0f}"
      f"  ({100*tot_inside/tot_valid:.4f} %)")
print(f"  positions falling OUTSIDE the lon/lat bins         : {tot_valid - tot_inside:,.0f}")
print(f"  of these also inside the {n_obs_bins} obs bins            : {tot_binned:,.0f}")
print(f"sum of the saved histogram H                         : {H.sum():,.0f}")
print(f"sum of the age-map counts                            : {A_cnt.sum():,.0f}")
print()
ok_h = abs(H.sum() - tot_binned) < 1
ok_a = abs(A_cnt.sum() - tot_inside) < 1
print("CHECK histogram total == binned positions read :", ok_h)
print("CHECK age-map count   == in-range positions    :", ok_a)
assert ok_h and ok_a, "histogram totals do not match the particles read!"

# releases per year, as a sanity check on the coverage of the climatology
print()
print(log.groupby(log.release.dt.year).size().to_string())

stores processed              : 1534
trajectories read             : 15,485,167
stores with >1 release time   : 0
obs lengths on disk           : [740, 925]
obs steps used per store      : [720]

finite (lon,lat) positions read (obs>=1)     : 11,148,916,804
  of these inside the lon/lat bins (counted by xhist): 11,148,916,375  (100.0000 %)
  positions falling OUTSIDE the lon/lat bins         : 429
  of these also inside the 36 obs bins            : 11,148,916,375
sum of the saved histogram H                         : 11,148,916,375
sum of the age-map counts                            : 11,148,916,375

CHECK histogram total == binned positions read : True
CHECK age-map count   == in-range positions    : True

release
1993    72
1994    73
1995    74
1996    73
1997    73
1998    73
1999    73
2000    74
2001    73
2002    73
2003    73
2004    73
2005    73
2006    73
2007    73
2008    73
2009    73
2010    73
2011    73
2012    73
2013    73


## Build the xarray products and save

* `_hist_year_month_v3.nc` — master array, counts by release year, release month,
  5-day obs bin, lon, lat.
* `_hist_month_v3.nc` — release-month climatology (sum over years), the direct
  equivalent of the template's `_hist_month.nc`; variable `histogram_lon_lat`.
* `_hist_season_v3.nc` — same but grouped into OND / JFM / AMJ / JAS.
* `_hist_age_month_v3.nc` — mean particle age [days] per cell and release month.

In [9]:
coords_ll = dict(lon=("lon", lon_c), lat=("lat", lat_c))
obs_c = np.arange(n_obs_bins)
# centre of each 5-day obs bin, in days since release
days_c = obs_c*obs_coarsen/4.0 + (obs_start + (obs_coarsen-1)/2.0)/4.0

attrs_common = dict(
    title="Spatial-bin analysis of the Mercator Amazon LPT production run 1993-2013",
    source=", ".join(tracks_dirs),
    n_stores=len(stores), n_trajectories=int(tot_traj),
    bin_lon_deg=dlon, bin_lat_deg=dlat, bins_file=bins_file,
    grid=("shared run_2/analysis grid: linspace bin edges from the data extremes, "
          "80 lon edges / 30 lat edges (v2.bins_v3.npz)"),
    obs_bin_days=obs_coarsen/4.0, obs_start=obs_start,
    note=("counts of 6-hourly particle positions; obs 0 (release position) skipped; "
          "obs aggregated into %d-day bins" % (obs_coarsen/4)),
    created=pd.Timestamp.now().isoformat(),
)

ym = xr.DataArray(H.astype("int32"),
        dims=("start_year","start_month","obs","lon","lat"),
        coords=dict(start_year=years, start_month=months, obs=obs_c, **coords_ll),
        name="histogram_lon_lat")
ym = ym.assign_coords(days_since_release=("obs", days_c))
ds_ym = ym.to_dataset()
ds_ym.attrs.update(attrs_common)

hm = xr.DataArray(H.sum(0).astype("int64"),
        dims=("start_month","obs","lon","lat"),
        coords=dict(start_month=months, obs=obs_c, **coords_ll),
        name="histogram_lon_lat").assign_coords(days_since_release=("obs", days_c))
ds_m = hm.to_dataset(); ds_m.attrs.update(attrs_common)

season_of_month = xr.DataArray([month_to_season[m] for m in months],
                               dims="start_month", coords=dict(start_month=months))
hs = hm.groupby(season_of_month.rename("start_season")).sum().astype("int64")
hs = hs.sel(start_season=seasons).rename("histogram_lon_lat")
ds_s = hs.to_dataset(); ds_s.attrs.update(attrs_common)

mean_age = xr.DataArray(np.where(A_cnt > 0, A_sum/np.where(A_cnt > 0, A_cnt, 1), np.nan
                                 ).astype("float32"),
        dims=("start_month","lon","lat"),
        coords=dict(start_month=months, **coords_ll), name="mean_age_days")
ds_a = xr.Dataset(dict(mean_age_days=mean_age,
                       n_positions=(("start_month","lon","lat"), A_cnt.astype("float32"))))
ds_a.attrs.update(attrs_common)
# NB: no CF "units" attribute here on purpose - "days" would make xarray decode
# the variable as a timedelta on reading.
ds_a.mean_age_days.attrs.update(
        long_name="mean age in days of the particles occupying the cell")

for name, dset in [(out_year_month, ds_ym), (out_month, ds_m),
                   (out_season, ds_s), (out_age, ds_a)]:
    p = os.path.join(out_dir, name)
    enc = {v: dict(zlib=True, complevel=4) for v in dset.data_vars}
    dset.to_netcdf(p, encoding=enc)
    print(f"{p}   {os.path.getsize(p)/1e6:8.1f} MB   "
          f"{dict(dset.sizes)}")

log.to_csv(os.path.join(out_dir, "_store_log_v3.csv"), index=False)
print("\nTOTAL notebook wall time: %.1f min" % ((time.time()-t_start)/60))

/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/_hist_year_month_v3.nc        5.2 MB   {'start_year': 21, 'start_month': 12, 'obs': 36, 'lon': 79, 'lat': 29}
/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/_hist_month_v3.nc        0.4 MB   {'start_month': 12, 'obs': 36, 'lon': 79, 'lat': 29}
/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/_hist_season_v3.nc        0.2 MB   {'obs': 36, 'lon': 79, 'lat': 29, 'start_season': 4}
/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/_hist_age_month_v3.nc        0.1 MB   {'start_month': 12, 'lon': 79, 'lat': 29}



TOTAL notebook wall time: 1.8 min


In [10]:
# quick numeric read-back check of the saved monthly file
chk = xr.open_dataset(os.path.join(out_dir, out_month))
print(chk)
saved = int(chk.histogram_lon_lat.sum())
print("\nsum over the saved monthly histogram:", f"{saved:,}")
print("CHECK saved monthly file total == positions binned:", saved == tot_binned)
assert saved == tot_binned
print("counts per release month:")
print(chk.histogram_lon_lat.sum(("obs","lon","lat")).to_series().to_string())

<xarray.Dataset> Size: 8MB
Dimensions:             (start_month: 12, obs: 36, lon: 79, lat: 29)
Coordinates:
  * start_month         (start_month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * obs                 (obs) int64 288B 0 1 2 3 4 5 6 ... 29 30 31 32 33 34 35
  * lon                 (lon) float64 632B -96.42 -95.38 ... -16.23 -15.19
  * lat                 (lat) float64 232B -3.757 -2.502 -1.246 ... 30.14 31.4
    days_since_release  (obs) float64 288B ...
Data variables:
    histogram_lon_lat   (start_month, obs, lon, lat) int64 8MB ...
Attributes:
    title:           Spatial-bin analysis of the Mercator Amazon LPT producti...
    source:          /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/tra...
    n_stores:        1534
    n_trajectories:  15485167
    bin_lon_deg:     1.041377669661373
    bin_lat_deg:     1.2554511689102739
    bins_file:       /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/ana...
    grid:            shared run_2/analysis grid: linspace bin 